#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("schemaSilver", "silver")
dbutils.widgets.text("schemaGold", "gold")
dbutils.widgets.text("catalogo", "project_dev")

In [0]:
schemaSilver = dbutils.widgets.get("schemaSilver")
schemaGold = dbutils.widgets.get("schemaGold")
catalogo = dbutils.widgets.get("catalogo")

# definir rutas
path_target = f"{catalogo}.{schemaGold}"

In [0]:
from pyspark.sql.functions import col, broadcast
from pyspark.sql import functions as F

##orders

In [0]:

df_orders = spark.table( f"{catalogo}.{schemaSilver}.orders_transform_end ")

## Agrupacion final

In [0]:
### seleccionar columnas necesarias y ahtupar
df_orders = (
    df_orders.select(
        "order_id",
        "order_item_id",
        "product_id",
        F.upper("order_status").alias("order_status"),
        F.col("order_delivered_carrier_date").cast("date").alias("order_delivered_carrier_date"),
        F.upper("product_category_name").alias("product_category_name"),
        F.upper("product_category_name_english").alias("product_category_name_english"),
        F.upper("seller_city").alias("seller_city"),
        "seller_state",
        F.upper("customer_city").alias("customer_city"),
        "customer_state",
        F.upper("payment_type").alias("payment_type"),
        "price",
        "freight",
        "customer_id",
        "seller_id"
    )
    .groupBy("order_id",
        "order_item_id",
        "product_id",
        "order_status",
        "order_delivered_carrier_date",
        "product_category_name",
        "product_category_name_english",
        "seller_city",
        "seller_state",
        "customer_city",
        "customer_state",
        "payment_type",
        "customer_id",
        "seller_id")
    .agg(
        F.sum("price").alias("price"),
        F.sum("freight").alias("freight")
    )
    .select("order_id",
        "order_item_id",
        "product_id",
        "order_status",
        "order_delivered_carrier_date",
        "product_category_name",
        "product_category_name_english",
        "seller_city",
        "seller_state",
        "customer_city",
        "customer_state",
        "payment_type",
        "price",
        "freight",
        "customer_id",
        "seller_id")
)

## cargar tabla

In [0]:
### guardar la tabla clean en el target
df_orders.write.mode("overwrite").insertInto(f"{path_target}.orders")

In [0]:
%sql
---- aplicar optimizer a la tabla target
OPTIMIZE ${catalogo}.${schemaGold}.orders
ZORDER BY order_id

In [0]:
%sql
--- habilitar vacum dinamico
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
---- solo tener archivos inferiores  24 horas
VACUUM ${catalogo}.${schemaGold}.orders RETAIN 24 HOURS

In [0]:
%sql
--- describir la historia de la tabla
DESCRIBE HISTORY ${catalogo}.${schemaGold}.orders